# 02 - Limpiar los viajes: tabla silver

Lee `viajes_bronze` (donde todo es texto) y arma `viajes_silver`, con los datos limpios. Hace cuatro cosas:

1. **Pone cada columna con su tipo correcto.** `fecha_hora` pasa a ser fecha y hora, `cantidad_pasajeros` y `ordinal_de_tramo` pasan a ser números enteros, `con_tarjeta` pasa a ser verdadero o falso, y a los textos se les sacan los espacios de más.
2. **Agrega columnas nuevas** a partir de la fecha: `fecha`, `hora`, `dia_semana` (1 es lunes y 7 es domingo) y `es_fin_de_semana`.
3. **Borra las filas que están mal:** fecha que no se puede leer, fecha de otro mes distinto al del archivo, pasajeros vacíos o menores a 1, o línea vacía.
4. **Borra las filas repetidas** (filas exactamente iguales a otra).

Cuántas filas se borran por cada motivo queda guardado por mes en la tabla `calidad_silver`. Así lo podemos contar en el informe.

Se procesa **un mes por vez**. Los meses que ya están en silver se saltean, salvo que pongas `FORZAR_SILVER = True`.

In [0]:
%run ./00_setup

## Opciones

In [0]:
MESES = None     # None quiere decir todos los meses que están en bronze
FORZAR_SILVER = False

In [0]:
import importlib

from pyspark.sql import functions as F

from bondis import silver

importlib.reload(silver)

silver.configurar_sesion(spark, config.TIMEZONE)
print("Zona horaria:", spark.conf.get("spark.sql.session.timeZone"))
print("Política de parseo de fechas:", spark.conf.get("spark.sql.legacy.timeParserPolicy"))


def mes_elegido(anio_mes):
    return MESES is None or anio_mes in MESES

## 1. Prueba antes de empezar: ¿se pueden leer las fechas?

Toma 10.000 filas del primer mes y trata de convertir el texto de la fecha en una fecha de verdad.
Si menos del 99 % se puede convertir, el notebook se frena y muestra ejemplos de cómo vienen escritas las fechas.
En ese caso hay que agregar esa forma de escribir la fecha en `silver.FORMATOS_FECHA`.

Es mejor que falle acá, con 10.000 filas, que después de procesar millones.

In [0]:
bronze = spark.table(config.TABLE_BRONZE)
meses_bronze = sorted(r["anio_mes"] for r in bronze.select("anio_mes").distinct().collect())
meses_a_revisar = [am for am in meses_bronze if mes_elegido(am)]
assert meses_a_revisar, f"Ningún mes de MESES está en bronze. Meses disponibles: {meses_bronze}"

muestra = bronze.where(F.col("anio_mes") == meses_a_revisar[0]).limit(10000)
prueba = (
    silver.tipar(muestra)
    .agg(F.count("*").alias("filas"), F.count("fecha_hora").alias("fechas_ok"))
    .first()
)
tasa = prueba["fechas_ok"] / prueba["filas"]
print(f"Muestra de {meses_a_revisar[0]}: {prueba['fechas_ok']:,} de {prueba['filas']:,} fechas convertidas ({tasa:.1%})")

if tasa < 0.99:
    display(muestra.select("fecha_evento").limit(20))
    raise ValueError(
        "Menos del 99 % de las fechas se pudo convertir. Revisar el formato de 'fecha_evento' "
        "en la tabla de arriba y agregarlo a silver.FORMATOS_FECHA."
    )

## 2. ¿Qué meses faltan procesar?

In [0]:
ya_en_silver = set()
if spark.catalog.tableExists(config.TABLE_SILVER):
    ya_en_silver = {
        r["anio_mes"] for r in spark.table(config.TABLE_SILVER).select("anio_mes").distinct().collect()
    }

pendientes = [am for am in meses_a_revisar if FORZAR_SILVER or am not in ya_en_silver]
print(f"Meses en bronze:          {len(meses_bronze)}")
print(f"Meses ya en silver:       {len(ya_en_silver & set(meses_a_revisar))}")
print(f"Meses a procesar ahora:   {len(pendientes)}  {pendientes}")

## 3. Limpiar mes por mes

Para cada mes:
1. Pone los tipos correctos (`tipar`).
2. Marca qué filas pasan cada control (`marcar_calidad`).
3. Cuenta cuántas filas fallan cada control.
4. Se queda con las filas buenas y saca las repetidas (`limpiar`).
5. Guarda el mes en `viajes_silver`, reemplazando solo ese mes.
6. Guarda los conteos del mes en `calidad_silver`.

In [0]:
def escribir_particion(df, tabla, anio_mes):
    """Guarda solo el mes anio_mes en la tabla, sin tocar los otros meses. Si la tabla no existe, la crea."""
    escritor = df.write.format("delta").mode("overwrite").option("mergeSchema", "true")
    if spark.catalog.tableExists(tabla):
        escritor.option("replaceWhere", f"anio_mes = '{anio_mes}'").saveAsTable(tabla)
    else:
        escritor.partitionBy("anio_mes").saveAsTable(tabla)


for am in pendientes:
    marcado = silver.marcar_calidad(silver.tipar(bronze.where(F.col("anio_mes") == am)))

    resumen = silver.resumen_calidad(marcado).first().asDict()
    limpio = silver.limpiar(marcado)
    escribir_particion(limpio, config.TABLE_SILVER, am)

    filas_silver = spark.table(config.TABLE_SILVER).where(F.col("anio_mes") == am).count()
    resumen.update(
        anio_mes=am,
        duplicados=resumen["filas_validas"] - filas_silver,
        filas_silver=filas_silver,
        pct_descartado=round(100 * (1 - filas_silver / resumen["filas_bronze"]), 3),
    )
    escribir_particion(spark.createDataFrame([resumen]), config.TABLE_CALIDAD_SILVER, am)

    print(f"[{am}] bronze {resumen['filas_bronze']:>12,}  ->  silver {filas_silver:>12,}  "
          f"({resumen['pct_descartado']:.3f} % descartado)")

## 4. Cuántas filas se borraron y por qué

Una fila por mes. Qué quiere decir cada columna:

| Columna | Por qué se borró la fila |
|---|---|
| `fecha_invalida` | La fecha no se pudo leer |
| `fecha_fuera_de_mes` | La fecha es de otro mes, no del mes del archivo |
| `pasajeros_invalidos` | `cantidad_pasajeros` está vacía, no es un número o es menor a 1 |
| `linea_nula` | La línea está vacía |
| `duplicados` | La fila era igual a otra |

Una misma fila puede fallar en más de un control. Por eso, si sumás los motivos, te puede dar más que el total de filas borradas.

In [0]:
if spark.catalog.tableExists(config.TABLE_CALIDAD_SILVER):
    display(
        spark.table(config.TABLE_CALIDAD_SILVER)
        .select("anio_mes", "filas_bronze", "fecha_invalida", "fecha_fuera_de_mes",
                "pasajeros_invalidos", "linea_nula", "duplicados", "filas_silver",
                "pct_descartado", "pasajeros_validos")
        .orderBy("anio_mes")
    )

## 5. Revisión de silver

Primero se muestran las columnas y sus tipos. Después, los pasajeros por hora en días de semana y fines de semana.

In [0]:
s = spark.table(config.TABLE_SILVER)
s.printSchema()

In [0]:
# Pasajeros por hora del primer mes procesado, separando días de semana y fines de semana.
# Si los datos están bien, en días de semana se tienen que ver dos picos: a la mañana y a la tarde.
mes_ejemplo = (pendientes or meses_a_revisar)[0]
display(
    s.where(F.col("anio_mes") == mes_ejemplo)
    .groupBy("es_fin_de_semana", "hora")
    .agg(F.sum("cantidad_pasajeros").alias("pasajeros"))
    .orderBy("es_fin_de_semana", "hora")
)

In [0]:
display(s.where(F.col("anio_mes") == mes_ejemplo).limit(10))

## 6. ¿Cómo identificamos una línea?

En los datos cada línea aparece con un nombre (`dsc_linea`, por ejemplo "103") y con un código (`linea_codigo`).
En gold agrupamos por **nombre**, porque es lo que la gente conoce. Las celdas de abajo revisan si eso trae problemas:

1. Si un mismo nombre tiene varios códigos o variantes (recorridos distintos de la misma línea).
2. Cuántos días funcionó cada línea en cada año. Sirve para ver líneas nuevas o que dejaron de andar.
3. Si un mismo código tiene más de un nombre.

In [0]:
from pyspark.sql import functions as F

s = spark.table(config.TABLE_SILVER).withColumn("anio", F.year("fecha"))

# 1. ¿Un mismo nombre de línea tiene varios códigos?
display(s.groupBy("dsc_linea")
         .agg(F.countDistinct("linea_codigo").alias("codigos"),
              F.countDistinct("sevar_codigo").alias("variantes"),
              F.sum("cantidad_pasajeros").alias("pasajeros"))
         .orderBy(F.desc("codigos"), F.desc("pasajeros")))

# 2. Días que funcionó cada línea en cada año (sirve para ver líneas nuevas o que se cerraron)
display(s.groupBy("dsc_linea")
         .pivot("anio", [2024, 2025, 2026])
         .agg(F.countDistinct("fecha"))
         .orderBy("dsc_linea"))

In [0]:
# 3. ¿Un mismo código tiene más de un nombre?
display(spark.table(config.TABLE_SILVER)
        .groupBy("linea_codigo")
        .agg(F.countDistinct("dsc_linea").alias("nombres"),
             F.collect_set("dsc_linea").alias("cuales"))
        .where("nombres > 1"))